# exp_004 — Agent context growth

This notebook analyzes measured agent trajectories only. Fixture smoke output is harness validation and is rejected before plotting.

All conclusions are descriptive and condition-limited.

In [ ]:
import csv
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from llm_lab.analysis import (
    aggregate_agent_trials,
    plot_reliability_by_length,
    plot_reliability_by_position,
    require_measured_trials,
    validate_complete_matrix,
)
from llm_lab.evaluation import load_trial_results

RESULTS = ROOT / 'experiments/exp_004-agent_context_growth/results'
MANIFEST_PATH = RESULTS / 'manifests/main.json'
RAW_PATH = RESULTS / 'raw/trials.jsonl'
PROCESSED_PATH = RESULTS / 'processed/summary.csv'
for required_path in (MANIFEST_PATH, RAW_PATH, PROCESSED_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f'measured exp_004 input is required: {required_path}')

run_manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
if run_manifest.get('fixture_only'):
    raise ValueError('fixture_only exp_004 results cannot be reported as Qwen measurements')
source_manifest = Path(run_manifest['source_manifest']['path'])
if not source_manifest.is_absolute():
    source_manifest = ROOT / source_manifest
if not source_manifest.is_file():
    raise FileNotFoundError(f'resolved source_manifest is required: {source_manifest}')

with PROCESSED_PATH.open(newline='', encoding='utf-8') as source:
    processed_rows = list(csv.DictReader(source))
if not processed_rows:
    raise ValueError('processed exp_004 summary.csv must contain measured rows')

measured_trials = require_measured_trials(load_trial_results(RAW_PATH))
rows = aggregate_agent_trials(measured_trials)
protocol = run_manifest['protocol']
variant_ids = [item['condition_id'] for item in run_manifest['source_manifest']['variants']]
task_types = protocol['task_types']
validate_complete_matrix(
    rows,
    variant_ids=variant_ids,
    trajectory_lengths=protocol['trajectory_lengths'],
    critical_positions=protocol['critical_positions'],
    task_types=task_types,
)

# Rows retain final_task_success, critical_fact_reuse_rate, and trajectory_context_tokens.
failure_category_counts = {
    (row['variant_condition_id'], row['trajectory_length'], row['requested_critical_position']): row['failure_category_counts']
    for row in rows
}
plot_reliability_by_length(rows, RESULTS / 'figures/reliability-vs-trajectory-length.png')
plot_reliability_by_position(rows, RESULTS / 'figures/reliability-vs-critical-position.png')
rows


## Interpretation checklist

Inspect final task success and critical-fact reuse against trajectory/context length and critical-information position. Report tool-call validity, repeated actions, recoveries, total input tokens, and `failure_category_counts` separately. Distinguish retrieval/state-tracking from tool/planning failures only when the logged trajectory makes the distinction observable. Do not call fixture results model findings.